In [1]:
import pandas as pd
import numpy as np

patients = pd.read_csv("patients.csv")
encounters = pd.read_csv("encounters.csv")
procedures = pd.read_csv("procedures.csv")
payers = pd.read_csv("payers.csv")
organizations = pd.read_csv("organizations.csv")
data_dict = pd.read_csv("data_dictionary.csv")

# STADARDIZING COLUMN NAMES

In [2]:
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )
    return df

patients = standardize_columns(patients)
encounters = standardize_columns(encounters)
procedures = standardize_columns(procedures)
organizations = standardize_columns(organizations)
payers = standardize_columns(payers)
data_dict = standardize_columns(data_dict)

In [3]:
print(patients.columns.tolist())
print(encounters.columns.tolist())
print(procedures.columns.tolist())
print(payers.columns.tolist())
print(data_dict.columns.tolist())
print(organizations.columns.tolist())

['id', 'birthdate', 'deathdate', 'prefix', 'first', 'last', 'suffix', 'maiden', 'marital', 'race', 'ethnicity', 'gender', 'birthplace', 'address', 'city', 'state', 'county', 'zip', 'lat', 'lon']
['id', 'start', 'stop', 'patient', 'organization', 'payer', 'encounterclass', 'code', 'description', 'base_encounter_cost', 'total_claim_cost', 'payer_coverage', 'reasoncode', 'reasondescription']
['start', 'stop', 'patient', 'encounter', 'code', 'description', 'base_cost', 'reasoncode', 'reasondescription']
['id', 'name', 'address', 'city', 'state_headquartered', 'zip', 'phone']
['table', 'field', 'description']
['id', 'name', 'address', 'city', 'state', 'zip', 'lat', 'lon']


# RENAMING COLUMNS

In [7]:
# Patients Table
patients = patients.rename(columns={
    "id": "patient_id",
    "birthdate": "birth_date",
    "deathdate": "death_date",
    "prefix": "name_prefix",
    "first": "first_name",
    "last": "last_name",
    "suffix": "name_suffix",
    "maiden": "maiden_name",
    "marital": "marital_status",
    "birthplace": "birth_place",
    "zip": "zip_code",
    "lat": "latitude",
    "lon": "longitude"
})

In [8]:
# Encounters Table
encounters = encounters.rename(columns={
    "id": "encounter_id",
    "start": "start_time",
    "stop": "end_time",
    "patient": "patient_id",
    "organization": "organization_id",
    "payer": "payer_id",
    "encounterclass": "encounter_class",
    "code": "encounter_code",
    "base_encounter_cost": "base_encounter_cost",
    "reasoncode": "reason_code",
    "reasondescription": "reason_description"
})

In [9]:
#procedures table
procedures = procedures.rename(columns={
    "start": "start_time",
    "stop": "end_time",
    "patient": "patient_id",
    "encounter": "encounter_id",
    "code": "procedure_code",
    "description": "procedure_description",
    "base_cost": "base_cost",
    "reasoncode": "reason_code",
    "reasondescription": "reason_description"
})

In [10]:
payers = payers.rename(columns={
    "id": "payer_id",
    "name": "payer_name",
    "state_headquartered": "state_headquartered",
    "zip": "zip_code",
    "phone": "phone"
})

In [11]:
organizations = organizations.rename(columns={
   
    "lat": "latitude",
    "lon": "longitude",
    "zip": "zip_code"
})
                                     

# Convert to proper datetime

In [12]:
encounters["start_time"] = pd.to_datetime(encounters["start_time"], errors="coerce")
encounters["end_time"] = pd.to_datetime(encounters["end_time"], errors="coerce")

procedures["start_time"] = pd.to_datetime(procedures["start_time"], errors="coerce")
procedures["end_time"] = pd.to_datetime(procedures["end_time"], errors="coerce")

# Split into Date and Time

In [13]:
# For Encounters
encounters["start_date"] = encounters["start_time"].dt.date
encounters["start_time_only"] = encounters["start_time"].dt.time

encounters["end_date"] = encounters["end_time"].dt.date
encounters["end_time_only"] = encounters["end_time"].dt.time

In [14]:
# For Procedures
procedures["start_date"] = procedures["start_time"].dt.date
procedures["start_time_only"] = procedures["start_time"].dt.time

procedures["end_date"] = procedures["end_time"].dt.date
procedures["end_time_only"] = procedures["end_time"].dt.time

In [16]:
encounters["start_hour"] = encounters["start_time"].dt.hour

# Duration in HOURS

In [17]:
# For Encounters
encounters["duration_hours"] = (
    encounters["end_time"] - encounters["start_time"]
).dt.total_seconds() / 3600

In [18]:
encounters["duration_hours"].describe()

count    27891.000000
mean         7.265995
std        398.323624
min          0.250000
25%          0.250000
50%          0.250000
75%          0.864167
max      44930.000000
Name: duration_hours, dtype: float64

In [19]:
#For Procedures
procedures["duration_hours"] = (
    procedures["end_time"] - procedures["start_time"]
).dt.total_seconds() / 3600

In [20]:
encounters["duration_category"] = pd.cut(
    encounters["duration_hours"],
    bins=[0, 1, 5, 24, 100, 1000],
    labels=["<1 hr", "1-5 hrs", "5-24 hrs", "1-4 days", "Long stay"]
)

In [21]:
procedures["duration_hours"].describe()

count    47701.000000
mean         0.554388
std          1.923217
min          0.000000
25%          0.250000
50%          0.275556
75%          0.493056
max        384.000000
Name: duration_hours, dtype: float64

# IDENTIFY OUTLIERS

In [22]:
# For Encounters
encounters.sort_values("duration_hours", ascending=False).head(3)

,encounter_id,start_time,end_time,patient_id,organization_id,payer_id,encounter_class,encounter_code,description,base_encounter_cost,...,payer_coverage,reason_code,reason_description,start_date,start_time_only,end_date,end_time_only,start_hour,duration_hours,duration_category
9704,fc2c2add-0b55-e300-5cea-8f43044eb330,2014-12-17 11:49:08+00:00,2020-02-01 13:49:08+00:00,5d47b9ac-3f37-f400-1747-c6c6f92130aa,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,390906007,Follow-up encounter (procedure),85.55,...,0.00,NaN,NaN,2014-12-17,11:49:08,2020-02-01,13:49:08,11,44930.000000,NaN
7735,2cb41a50-819c-ef7e-87df-b5b47b2e01a1,2014-03-27 00:33:14+00:00,2019-03-13 15:33:14+00:00,09e0ec41-cd4c-50b2-b425-824013777beb,d78e84ec-30aa-3bba-a33a-f29a3a454662,6e2f1a2d-27bd-3701-8d08-dae202c58632,ambulatory,390906007,Follow-up encounter (procedure),85.55,...,7.91,NaN,NaN,2014-03-27,00:33:14,2019-03-13,15:33:14,0,43503.000000,NaN
7805,7a19a260-e4a9-7b81-3f93-e07144d634d9,2014-04-02 22:26:28+00:00,2015-07-02 13:44:57+00:00,8885aba5-afce-9e3b-0c1d-a684a347b5fb,d78e84ec-30aa-3bba-a33a-f29a3a454662,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,ambulatory,390906007,Follow-up encounter (procedure),85.55,...,24.27,NaN,NaN,2014-04-02,22:26:28,2015-07-02,13:44:57,22,10935.308056,NaN


# Defining THRESHOLD & Filter out extreme outliers

In [23]:
MAX_DURATION_HOURS = 720

encounters = encounters[encounters["duration_hours"] <= MAX_DURATION_HOURS]

In [24]:
encounters["duration_hours"].describe()

count    27878.000000
mean         1.835653
std         10.296424
min          0.250000
25%          0.250000
50%          0.250000
75%          0.862431
max        659.000000
Name: duration_hours, dtype: float64

In [25]:
encounters.head(2)

,encounter_id,start_time,end_time,patient_id,organization_id,payer_id,encounter_class,encounter_code,description,base_encounter_cost,...,payer_coverage,reason_code,reason_description,start_date,start_time_only,end_date,end_time_only,start_hour,duration_hours,duration_category
0,32c84703-2481-49cd-d571-3899d5820253,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,3de74169-7f67-9304-91d4-757e0f3a14d2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,185347001,Encounter for problem (procedure),85.55,...,0.0,NaN,NaN,2011-01-02,09:26:36,2011-01-02,12:58:36,9,3.533333,1-5 hrs
1,c98059da-320a-c0a6-fced-c8815f3e3f39,2011-01-03 05:44:39+00:00,2011-01-03 06:01:42+00:00,d9ec2e44-32e9-9148-179a-1653348cc4e2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,outpatient,308335008,Patient encounter procedure,142.58,...,0.0,NaN,NaN,2011-01-03,05:44:39,2011-01-03,06:01:42,5,0.284167,<1 hr


In [26]:
patients.head(2)

,patient_id,birth_date,death_date,name_prefix,first_name,last_name,name_suffix,maiden_name,marital_status,race,ethnicity,gender,birth_place,address,city,state,county,zip_code,latitude,longitude
0,5605b66b-e92d-c16c-1b83-b8bf7040d51f,1977-03-19,NaN,Mrs.,Nikita578,Erdman779,NaN,Leannon79,M,white,nonhispanic,F,Wakefield Massachusetts US,510 Little Station Unit 69,Quincy,Massachusetts,Norfolk County,2186.0,42.290937,-70.975503
1,6e5ae27c-8038-7988-e2c0-25a103f01bfa,1940-02-19,NaN,Mr.,Zane918,Hodkiewicz467,NaN,NaN,M,white,nonhispanic,M,Brookline Massachusetts US,747 Conn Throughway,Boston,Massachusetts,Suffolk County,2135.0,42.308831,-71.063162


# CALCULATE AGE OF THE PATIENTS

In [27]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      974 non-null    object 
 1   birth_date      974 non-null    object 
 2   death_date      154 non-null    object 
 3   name_prefix     974 non-null    object 
 4   first_name      974 non-null    object 
 5   last_name       974 non-null    object 
 6   name_suffix     21 non-null     object 
 7   maiden_name     386 non-null    object 
 8   marital_status  973 non-null    object 
 9   race            974 non-null    object 
 10  ethnicity       974 non-null    object 
 11  gender          974 non-null    object 
 12  birth_place     974 non-null    object 
 13  address         974 non-null    object 
 14  city            974 non-null    object 
 15  state           974 non-null    object 
 16  county          974 non-null    object 
 17  zip_code        832 non-null    flo

In [28]:
encounters.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27878 entries, 0 to 27890
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   encounter_id         27878 non-null  object             
 1   start_time           27878 non-null  datetime64[ns, UTC]
 2   end_time             27878 non-null  datetime64[ns, UTC]
 3   patient_id           27878 non-null  object             
 4   organization_id      27878 non-null  object             
 5   payer_id             27878 non-null  object             
 6   encounter_class      27878 non-null  object             
 7   encounter_code       27878 non-null  int64              
 8   description          27878 non-null  object             
 9   base_encounter_cost  27878 non-null  float64            
 10  total_claim_cost     27878 non-null  float64            
 11  payer_coverage       27878 non-null  float64            
 12  reason_code          83

# create latest encounter date per patient

In [29]:
print(encounters.columns.tolist())

['encounter_id', 'start_time', 'end_time', 'patient_id', 'organization_id', 'payer_id', 'encounter_class', 'encounter_code', 'description', 'base_encounter_cost', 'total_claim_cost', 'payer_coverage', 'reason_code', 'reason_description', 'start_date', 'start_time_only', 'end_date', 'end_time_only', 'start_hour', 'duration_hours', 'duration_category']


In [30]:
encounters["end_date"] = pd.to_datetime(encounters["end_date"], errors="coerce")
patients["birth_date"] = pd.to_datetime(patients["birth_date"], errors="coerce")
patients["death_date"] = pd.to_datetime(patients["death_date"], errors="coerce")

latest_encounter = (
    encounters.groupby("patient_id", as_index=False)["end_date"]
    .max()
)

latest_encounter = latest_encounter.rename(columns={"end_date": "latest_encounter_date"})

In [31]:
print(latest_encounter.head())
print(latest_encounter.columns.tolist())

                             patient_id latest_encounter_date
0  002bc307-2fff-04ba-161b-98cce123e226            2021-10-12
1  0034fe01-207f-275f-6b4b-821f7b0af044            2021-12-03
2  00abe029-00fa-a666-34f5-258a36978f6d            2021-09-19
3  010ff9c1-564b-6e32-09f4-29cb4224bba9            2017-11-02
4  01274098-150f-8211-6150-29f2a2da266c            2011-01-24
['patient_id', 'latest_encounter_date']


In [32]:
#merge with patients
patients = patients.merge(latest_encounter, on="patient_id", how="left")

In [33]:
print(patients.columns.tolist())

['patient_id', 'birth_date', 'death_date', 'name_prefix', 'first_name', 'last_name', 'name_suffix', 'maiden_name', 'marital_status', 'race', 'ethnicity', 'gender', 'birth_place', 'address', 'city', 'state', 'county', 'zip_code', 'latitude', 'longitude', 'latest_encounter_date']


In [34]:
patients["reference_date"] = patients["death_date"].fillna(patients["latest_encounter_date"])

In [35]:
#calculate age
import numpy as np

patients["age"] = np.floor(
    (patients["reference_date"] - patients["birth_date"]).dt.days / 365.25
).astype("Int64")

In [36]:
patients[["patient_id", "birth_date", "death_date", "latest_encounter_date", "reference_date", "age"]].head()

,patient_id,birth_date,death_date,latest_encounter_date,reference_date,age
0,5605b66b-e92d-c16c-1b83-b8bf7040d51f,1977-03-19,NaT,2020-11-08,2020-11-08,43
1,6e5ae27c-8038-7988-e2c0-25a103f01bfa,1940-02-19,NaT,2020-09-15,2020-09-15,80
2,8123d076-0886-9007-e956-d5864aa121a7,1958-06-04,NaT,2021-08-05,2021-08-05,63
3,770518e4-6133-648e-60c9-071eb2f0e2ce,1928-12-25,2017-09-29,2017-08-16,2017-09-29,88
4,f96addf5-81b9-0aab-7855-d208d3d352c5,1928-12-25,2014-02-23,2014-02-21,2014-02-23,85


In [37]:
patients.head(3)

,patient_id,birth_date,death_date,name_prefix,first_name,last_name,name_suffix,maiden_name,marital_status,race,...,address,city,state,county,zip_code,latitude,longitude,latest_encounter_date,reference_date,age
0,5605b66b-e92d-c16c-1b83-b8bf7040d51f,1977-03-19,NaT,Mrs.,Nikita578,Erdman779,NaN,Leannon79,M,white,...,510 Little Station Unit 69,Quincy,Massachusetts,Norfolk County,2186.0,42.290937,-70.975503,2020-11-08,2020-11-08,43
1,6e5ae27c-8038-7988-e2c0-25a103f01bfa,1940-02-19,NaT,Mr.,Zane918,Hodkiewicz467,NaN,NaN,M,white,...,747 Conn Throughway,Boston,Massachusetts,Suffolk County,2135.0,42.308831,-71.063162,2020-09-15,2020-09-15,80
2,8123d076-0886-9007-e956-d5864aa121a7,1958-06-04,NaT,Mr.,Quinn173,Marquardt819,NaN,NaN,M,white,...,816 Okuneva Extension Apt 91,Quincy,Massachusetts,Norfolk County,2170.0,42.265177,-70.967085,2021-08-05,2021-08-05,63


In [38]:
patients = patients.drop(columns=["latest_encounter_date_x", "latest_encounter_date_y"], errors="ignore")

In [39]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   patient_id             974 non-null    object        
 1   birth_date             974 non-null    datetime64[ns]
 2   death_date             154 non-null    datetime64[ns]
 3   name_prefix            974 non-null    object        
 4   first_name             974 non-null    object        
 5   last_name              974 non-null    object        
 6   name_suffix            21 non-null     object        
 7   maiden_name            386 non-null    object        
 8   marital_status         973 non-null    object        
 9   race                   974 non-null    object        
 10  ethnicity              974 non-null    object        
 11  gender                 974 non-null    object        
 12  birth_place            974 non-null    object        
 13  addre

# DUPLICATE HANDLING

In [40]:
# Full row duplicates
print("Patients full duplicates:", patients.duplicated().sum())
print("Encounters full duplicates:", encounters.duplicated().sum())
print("Procedures full duplicates:", procedures.duplicated().sum())
print("Payers full duplicates:", payers.duplicated().sum())

Patients full duplicates: 0
Encounters full duplicates: 0
Procedures full duplicates: 0
Payers full duplicates: 0


In [42]:
# Primary key duplicates
print("Patients ID duplicates:", patients["patient_id"].duplicated().sum())
print("Encounters ID duplicates:", encounters["encounter_id"].duplicated().sum())
print("Payers ID duplicates:", payers["payer_id"].duplicated().sum())

Patients ID duplicates: 0
Encounters ID duplicates: 0
Payers ID duplicates: 0


In [43]:
# Patient's Table Business logic duplicates
patients_dup = patients.duplicated(
    subset=["first_name", "last_name", "birth_date", "gender"],
    keep=False
)

patients[patients_dup].sort_values(by=["last_name", "first_name"])

,patient_id,birth_date,death_date,name_prefix,first_name,last_name,name_suffix,maiden_name,marital_status,race,...,address,city,state,county,zip_code,latitude,longitude,latest_encounter_date,reference_date,age


In [44]:
# Encounters Table Business logic duplicates
encounters_dup = encounters.duplicated(
    subset=["patient_id", "start_time", "encounter_class"]
).sum()

print("Encounters business duplicates:", encounters_dup)

Encounters business duplicates: 27


In [45]:
# Remove Duplicates
encounters = encounters.drop_duplicates(
    subset=["patient_id", "start_time", "encounter_class"]
)

In [46]:
# Procedures Table Business Logic Duplicates
procedures_dup = procedures.duplicated(
    subset=["patient_id", "encounter_id", "procedure_code"]
).sum()

print("Procedures business duplicates:", procedures_dup)

Procedures business duplicates: 4323


In [47]:
# Removing Duplicates
procedures = procedures.drop_duplicates(
    subset=["patient_id", "encounter_id", "procedure_code"]
)

In [48]:
# payer's table
print("Payers full duplicates:", payers.duplicated().sum())

Payers full duplicates: 0


In [49]:
print("Payers ID duplicates:", payers["payer_id"].duplicated().sum())

Payers ID duplicates: 0


In [50]:
payers_dup = payers.duplicated(
    subset=["payer_name", "phone", "zip_code"],
    keep=False
)

payers[payers_dup].sort_values(by=["payer_name"])

,payer_id,payer_name,address,city,state_headquartered,zip_code,phone


# MISSING VALUES ANALYSIS

In [51]:
def missing_summary(df):
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    summary = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent
    }).sort_values(by="missing_percent", ascending=False)

    return summary

print("Patients:\n", missing_summary(patients), "\n")
print("Encounters:\n", missing_summary(encounters), "\n")
print("Procedures:\n", missing_summary(procedures), "\n")
print("Payers:\n", missing_summary(payers), "\n")

PATIENTS TABLE

In [52]:
patients = patients.drop(columns=["name_suffix"])

In [53]:
patients = patients.drop(columns=["maiden_name"])

In [54]:
patients["marital_status"] = patients["marital_status"].fillna("Unknown")

In [55]:
patients["zip_code"] = patients["zip_code"].fillna("Unknown")

ENCOUNTERS TABLE

In [56]:
# Important in HealthCare Data
# Many encounters don’t have a specific reason record

encounters["reason_code"] = encounters["reason_code"].fillna("Unknown")
encounters["reason_description"] = encounters["reason_description"].fillna("Unknown")

PROCEDURES TABLE

In [57]:
procedures["reason_code"] = procedures["reason_code"].fillna("Unknown")
procedures["reason_description"] = procedures["reason_description"].fillna("Unknown")

PAYERS TABLE

In [58]:
payers["address"] = payers["address"].fillna("Unknown")
payers["state_headquartered"] = payers["state_headquartered"].fillna("Unknown")
payers["city"] = payers["city"].fillna("Unknown")
payers["zip_code"] = payers["zip_code"].fillna("Unknown")
payers["phone"] = payers["phone"].fillna("Not Available")

In [59]:
print("Patients:\n", patients.isnull().sum(), "\n")
print("Encounters:\n", encounters.isnull().sum(), "\n")
print("Procedures:\n", procedures.isnull().sum(), "\n")
print("Payers:\n", payers.isnull().sum(), "\n")

Patients:
 patient_id                 0
birth_date                 0
death_date               820
name_prefix                0
first_name                 0
last_name                  0
marital_status             0
race                       0
ethnicity                  0
gender                     0
birth_place                0
address                    0
city                       0
state                      0
county                     0
zip_code                   0
latitude                   0
longitude                  0
latest_encounter_date      0
reference_date             0
age                        0
dtype: int64 

Encounters:
 encounter_id           0
start_time             0
end_time               0
patient_id             0
organization_id        0
payer_id               0
encounter_class        0
encounter_code         0
description            0
base_encounter_cost    0
total_claim_cost       0
payer_coverage         0
reason_code            0
reason_description     0
st

# Data Trimming, text cleaning 

In [60]:
patients["first_name"] = patients["first_name"].apply(
    lambda x: ''.join([ch for ch in str(x) if ch.isalpha()])
)

patients["last_name"] = patients["last_name"].apply(
    lambda x: ''.join([ch for ch in str(x) if ch.isalpha()])
)

In [61]:
patients["first_name"] = patients["first_name"].str.strip().str.title()
patients["last_name"] = patients["last_name"].str.strip().str.title()

patients["first_name"] = patients["first_name"].replace("", "Unknown")
patients["last_name"] = patients["last_name"].replace("", "Unknown")

In [62]:
patients[["first_name","last_name"]].head()

,first_name,last_name
0,Nikita,Erdman
1,Zane,Hodkiewicz
2,Quinn,Marquardt
3,Abel,Smitham
4,Edwin,Labadie


In [63]:
patients["gender"].unique()

array(['F', 'M'], dtype=object)

In [64]:
patients["gender"] = patients["gender"].map({
    "M": "Male",
    "F": "Female"
})

In [65]:
patients["gender"].head()

0    Female
1      Male
2      Male
3      Male
4      Male
Name: gender, dtype: object

In [66]:
patients["race"].unique()

array(['white', 'asian', 'black', 'native', 'hawaiian', 'other'],
      dtype=object)

In [67]:
patients["ethnicity"].unique()

array(['nonhispanic', 'hispanic'], dtype=object)

In [68]:
patients["race"] = patients["race"].str.strip().str.lower()
patients["ethnicity"] = patients["ethnicity"].str.strip().str.lower()

In [69]:
patients["race"] = patients["race"].replace({
    "white": "White",
    "asian": "Asian",
    "black": "Black",
    "native": "Native",
    "hawaiian": "Hawaiian",
    "other": "Other"
})

patients["ethnicity"] = patients["ethnicity"].replace({
    "hispanic": "Hispanic",
    "nonhispanic": "Non-Hispanic"
})

In [70]:
print(patients["race"].unique())

['White' 'Asian' 'Black' 'Native' 'Hawaiian' 'Other']


In [71]:
print(patients["ethnicity"].unique())

['Non-Hispanic' 'Hispanic']


In [72]:
patients["name_prefix"].unique()

array(['Mrs.', 'Mr.', 'Ms.'], dtype=object)

In [73]:
patients["name_prefix"] = patients["name_prefix"].str.strip()

patients["name_prefix"] = patients["name_prefix"].replace({
    "Mr.": "Mr",
    "Mrs.": "Mrs",
    "Ms.": "Ms",
    "Dr.": "Dr"
})

patients["name_prefix"] = patients["name_prefix"].fillna("Unknown")

In [74]:
patients["name_prefix"].unique()

array(['Mrs', 'Mr', 'Ms'], dtype=object)

In [75]:
location_cols = ["city", "state", "county", "birth_place"]

for col in location_cols:
    patients[col] = patients[col].astype("string").str.strip().str.title()

In [76]:
for col in location_cols:
    patients[col] = patients[col].fillna("Unknown")

In [77]:
patients[["city", "state", "county", "birth_place"]].head()

,city,state,county,birth_place
0,Quincy,Massachusetts,Norfolk County,Wakefield Massachusetts Us
1,Boston,Massachusetts,Suffolk County,Brookline Massachusetts Us
2,Quincy,Massachusetts,Norfolk County,Gardner Massachusetts Us
3,Boston,Massachusetts,Suffolk County,Randolph Massachusetts Us
4,Boston,Massachusetts,Suffolk County,Stow Massachusetts Us


For Encounter Table

In [78]:
encounters["encounter_class"].unique()

array(['ambulatory', 'outpatient', 'wellness', 'urgentcare', 'inpatient',
       'emergency'], dtype=object)

In [79]:
encounters["encounter_class"] = encounters["encounter_class"].replace({
    "ambulatory" : "Ambulatory",
    "outpatient" : "Outpatient",
    "wellness" : "Wellness",
    "urgentcare" : "Urgentcare",
    "inpatient" : "Inpatient",
    "emergency" : "Emergency",
})

encounters["encounter_class"].value_counts()

encounter_class
Ambulatory    12517
Outpatient     6283
Urgentcare     3665
Emergency      2322
Wellness       1931
Inpatient      1133
Name: count, dtype: int64

In [80]:
encounters.head(1)

,encounter_id,start_time,end_time,patient_id,organization_id,payer_id,encounter_class,encounter_code,description,base_encounter_cost,...,payer_coverage,reason_code,reason_description,start_date,start_time_only,end_date,end_time_only,start_hour,duration_hours,duration_category
0,32c84703-2481-49cd-d571-3899d5820253,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,3de74169-7f67-9304-91d4-757e0f3a14d2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,Ambulatory,185347001,Encounter for problem (procedure),85.55,...,0.0,Unknown,Unknown,2011-01-02,09:26:36,2011-01-02,12:58:36,9,3.533333,1-5 hrs


In [81]:
encounters["reason_description"] = (
    encounters["reason_description"]
    .astype("string")
    .str.strip()
)

In [82]:
encounters.columns.tolist()

['encounter_id',
 'start_time',
 'end_time',
 'patient_id',
 'organization_id',
 'payer_id',
 'encounter_class',
 'encounter_code',
 'description',
 'base_encounter_cost',
 'total_claim_cost',
 'payer_coverage',
 'reason_code',
 'reason_description',
 'start_date',
 'start_time_only',
 'end_date',
 'end_time_only',
 'start_hour',
 'duration_hours',
 'duration_category']

In [83]:
payer_text_cols = ["payer_name", "address", "city", "state_headquartered", "phone"]

for col in payer_text_cols:
    payers[col] = payers[col].astype("string").str.strip()

# Final Cleaning Stage - Sanity Check

In [84]:
#clean shapes of all cleaned tables

print("Patients shape:", patients.shape)
print("Encounters shape:", encounters.shape)
print("Procedures shape:", procedures.shape)
print("Payers shape:", payers.shape)

Patients shape: (974, 21)
Encounters shape: (27851, 21)
Procedures shape: (43378, 14)
Payers shape: (10, 7)


In [85]:
# Final NULL Check
def missing_summary(df):
    missing_count = df.isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    return pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_percent
    }).sort_values(by="missing_percent", ascending=False)

In [86]:
print("Patients\n", missing_summary(patients), "\n")
print("Encounters\n", missing_summary(encounters), "\n")
print("Procedures\n", missing_summary(procedures), "\n")
print("Payers\n", missing_summary(payers), "\n")

Patients
                        missing_count  missing_percent
death_date                       820        84.188912
patient_id                         0         0.000000
birth_date                         0         0.000000
name_prefix                        0         0.000000
first_name                         0         0.000000
last_name                          0         0.000000
marital_status                     0         0.000000
race                               0         0.000000
ethnicity                          0         0.000000
gender                             0         0.000000
birth_place                        0         0.000000
address                            0         0.000000
city                               0         0.000000
state                              0         0.000000
county                             0         0.000000
zip_code                           0         0.000000
latitude                           0         0.000000
longitude         

In [87]:
# Re-check key integrity

print("Encounters linked to valid patients:",
      encounters["patient_id"].isin(patients["patient_id"]).mean())

print("Procedures linked to valid encounters:",
      procedures["encounter_id"].isin(encounters["encounter_id"]).mean())

print("Encounters linked to valid payers:",
      encounters["payer_id"].isin(payers["payer_id"]).mean())

Encounters linked to valid patients: 1.0
Procedures linked to valid encounters: 0.9979713218682281
Encounters linked to valid payers: 1.0


In [88]:
# Re-check duplicates after cleaning

print("Patients full duplicates:", patients.duplicated().sum())
print("Encounters full duplicates:", encounters.duplicated().sum())
print("Procedures full duplicates:", procedures.duplicated().sum())
print("Payers full duplicates:", payers.duplicated().sum())

Patients full duplicates: 0
Encounters full duplicates: 0
Procedures full duplicates: 0
Payers full duplicates: 0


In [89]:
print("Patients ID duplicates:", patients["patient_id"].duplicated().sum())
print("Encounters ID duplicates:", encounters["encounter_id"].duplicated().sum())
print("Payers ID duplicates:", payers["payer_id"].duplicated().sum())

Patients ID duplicates: 0
Encounters ID duplicates: 0
Payers ID duplicates: 0


# Check important numerical columns


In [90]:
patients["age"].describe()

count        974.0
mean     67.442505
std      20.262874
min           21.0
25%           50.0
50%           69.0
75%           85.0
max           99.0
Name: age, dtype: Float64

In [91]:
patients[(patients["age"] < 0) | (patients["age"] > 120)]

,patient_id,birth_date,death_date,name_prefix,first_name,last_name,marital_status,race,ethnicity,gender,...,address,city,state,county,zip_code,latitude,longitude,latest_encounter_date,reference_date,age


In [92]:
encounters["duration_hours"].describe()

count    27851.000000
mean         1.836808
std         10.301336
min          0.250000
25%          0.250000
50%          0.250000
75%          0.861389
max        659.000000
Name: duration_hours, dtype: float64

In [93]:
encounters["base_encounter_cost"].describe()

count    27851.000000
mean       116.219206
std         28.406943
min         85.550000
25%         85.550000
50%        136.800000
75%        142.580000
max        146.180000
Name: base_encounter_cost, dtype: float64

In [94]:
encounters["total_claim_cost"].describe()

count     27851.000000
mean       3643.374898
std        9211.397464
min           0.000000
25%         142.580000
50%         278.580000
75%        1417.195000
max      641882.700000
Name: total_claim_cost, dtype: float64

In [95]:
encounters["payer_coverage"].describe()

count     27851.000000
mean       1116.163391
std        4771.808584
min           0.000000
25%           0.000000
50%          34.870000
75%         155.770000
max      247751.420000
Name: payer_coverage, dtype: float64

In [96]:
encounters[encounters["duration_hours"] < 0]
encounters[encounters["base_encounter_cost"] < 0]
encounters[encounters["total_claim_cost"] < 0]
encounters[encounters["payer_coverage"] < 0]

,encounter_id,start_time,end_time,patient_id,organization_id,payer_id,encounter_class,encounter_code,description,base_encounter_cost,...,payer_coverage,reason_code,reason_description,start_date,start_time_only,end_date,end_time_only,start_hour,duration_hours,duration_category


In [97]:
# Procedures cost and duration
procedures["duration_hours"].describe()

count    43378.000000
mean         0.583236
std          2.013507
min          0.000000
25%          0.250000
50%          0.343333
75%          0.500000
max        384.000000
Name: duration_hours, dtype: float64

In [98]:
procedures["base_cost"].describe()

count     43378.000000
mean       2361.973881
std        5571.584991
min           1.000000
25%         431.000000
50%         431.000000
75%        1203.000000
max      289531.000000
Name: base_cost, dtype: float64

In [99]:
procedures[procedures["duration_hours"] < 0]

,start_time,end_time,patient_id,encounter_id,procedure_code,procedure_description,base_cost,reason_code,reason_description,start_date,start_time_only,end_date,end_time_only,duration_hours


In [100]:
procedures[procedures["base_cost"] < 0]

,start_time,end_time,patient_id,encounter_id,procedure_code,procedure_description,base_cost,reason_code,reason_description,start_date,start_time_only,end_date,end_time_only,duration_hours


In [101]:
patients.to_csv("patients_clean.csv", index=False)
encounters.to_csv("encounters_clean.csv", index=False)
procedures.to_csv("procedures_clean.csv", index=False)
payers.to_csv("payers_clean.csv", index=False)
organizations.to_csv("organizations_clean.csv",index = False)